# História 2 — Comparativo entre Bairros de Perfil Parecido

**Objetivo:** Agrupar os bairros do Rio de Janeiro em clusters com perfil de criminalidade similar, permitindo ao morador comparar seu bairro com seus *pares* — não com toda a cidade.

**Recorte:**
- Cidade: Rio de Janeiro
- Período: histórico completo nos dados Gabriel (maio/2020 – maio/2026)
- Bairros incluídos: apenas os com ≥ 50 ocorrências registradas (histórico robusto o suficiente para estatísticas confiáveis)

**Algoritmo:** K-Means com k=4 (melhor silhouette balanceando interpretabilidade e tamanho mínimo de cluster)

**Features usadas (todas derivadas dos dados Gabriel):**
- Distribuição por categoria de crime (% Roubo, Furto, Tentativas, Acidentes, Golpes, Vandalismo)
- % de ocorrências no período noturno (18h–6h)
- % por gênero do local (Público, Residencial, Empresarial)
- Camaleões por ocorrência (proxy de densidade de cobertura)
- Taxa de efetividade da análise forense

## 0. Imports e configurações

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

import os
os.makedirs('outputs', exist_ok=True)

# Constantes
CIDADE     = 'Rio de Janeiro'
MIN_OCC    = 50      # bairros com menos ocorrências têm perfil estatístico pouco confiável
N_CLUSTERS = 4
RANDOM_STATE = 42

CORES_CLUSTER = {
    0: '#f59e0b',  # Amarelo — Eixo Viário
    1: '#3b82f6',  # Azul    — Zona Turística
    2: '#e94560',  # Vermelho — Urbano Consolidado
    3: '#22c55e',  # Verde   — Residencial Periurbano
}

NOMES_CLUSTER = {
    0: 'Eixo Viário',
    1: 'Zona Turística',
    2: 'Urbano Consolidado',
    3: 'Residencial Periurbano',
}

print('Imports OK')

## 1. Carregamento e filtragem dos dados

In [ ]:
occ_raw = pd.read_csv('ocorrencias.csv', parse_dates=['DataOcorrencia'])
sen_raw = pd.read_csv('sensores.csv')

occ_rj = occ_raw.query('Estado == "RJ" and Cidade == @CIDADE').copy()
sen_rj = sen_raw.query('Estado == "RJ" and Cidade == @CIDADE').copy()

print(f'Ocorrências RJ (total histórico): {len(occ_rj):,}')
print(f'Camaleões RJ: {sen_rj["IDDispositivo"].nunique():,}')

# Filtrar bairros com histórico robusto
vol_bairro    = occ_rj.groupby('Bairro').size()
bairros_ok    = vol_bairro[vol_bairro >= MIN_OCC].index
bairros_poucos = vol_bairro[vol_bairro < MIN_OCC].index

df = occ_rj[occ_rj['Bairro'].isin(bairros_ok)].copy()

print(f'\nBairros com >= {MIN_OCC} ocorrências: {len(bairros_ok)}')
print(f'Bairros excluídos (dados insuficientes): {len(bairros_poucos)}')
print(f'Ocorrências utilizadas: {len(df):,} ({len(df)/len(occ_rj)*100:.1f}% do total RJ)')

## 2. Engenharia de features

Cada bairro é representado por um vetor de features que captura:
- **Perfil de crime**: o que acontece (roubo, furto, acidente...)
- **Perfil temporal**: quando acontece (noturno vs. diurno)
- **Perfil espacial**: onde acontece (público, residencial, comercial)
- **Cobertura Gabriel**: densidade de Camaleões relativa às ocorrências
- **Qualidade das imagens**: taxa de efetividade da análise forense

In [ ]:
# Volume total por bairro
feat = df.groupby('Bairro').size().rename('total_occ').reset_index()

# 1. Distribuição por categoria de crime
CATS = ['Roubo', 'Furto', 'Tentativas', 'Acidentes de Trânsito',
        'Golpes e Fraudes', 'Vandalismo e Danos']
cat_pct = (
    df.groupby(['Bairro', 'CategoriaCrime']).size()
    .unstack(fill_value=0)
    .reindex(columns=CATS, fill_value=0)
    .apply(lambda r: r / r.sum() if r.sum() > 0 else r, axis=1)
    .rename(columns={
        'Roubo'               : 'pct_roubo',
        'Furto'               : 'pct_furto',
        'Tentativas'          : 'pct_tentativas',
        'Acidentes de Trânsito': 'pct_acidentes',
        'Golpes e Fraudes'    : 'pct_golpes',
        'Vandalismo e Danos'  : 'pct_vandalismo',
    })
    .reset_index()
)
feat = feat.merge(cat_pct, on='Bairro', how='left')

# 2. % noturno (18h–6h)
def is_noturno(intervalo):
    if pd.isna(intervalo): return False
    h = int(str(intervalo).split('h')[0].strip())
    return h >= 18 or h < 6

df['noturno'] = df['Intervalo'].apply(is_noturno)
feat = feat.merge(
    df.groupby('Bairro')['noturno'].mean().rename('pct_noturno').reset_index(),
    on='Bairro', how='left'
)

# 3. Distribuição por gênero do local
gen_pct = (
    df.groupby(['Bairro', 'GeneroLocal']).size()
    .unstack(fill_value=0)
    .apply(lambda r: r / r.sum() if r.sum() > 0 else r, axis=1)
    .rename(columns=str.lower)
    .add_prefix('pct_')
    .reset_index()
)
feat = feat.merge(gen_pct, on='Bairro', how='left')

# 4. Camaleões por ocorrência
cam = (
    sen_rj.groupby('Bairro')['IDDispositivo']
    .nunique().rename('n_camaleoes').reset_index()
)
feat = feat.merge(cam, on='Bairro', how='left')
feat['n_camaleoes'] = feat['n_camaleoes'].fillna(0)
feat['cam_por_occ'] = feat['n_camaleoes'] / feat['total_occ']

# 5. Taxa de efetividade
efet = (
    df.groupby('Bairro')['EfetividadeAnalise']
    .apply(lambda x: (x == 'Efetiva').mean())
    .rename('taxa_efetividade').reset_index()
)
feat = feat.merge(efet, on='Bairro', how='left')
feat = feat.set_index('Bairro').fillna(0)

FEAT_COLS = [c for c in feat.columns if c not in ['total_occ', 'n_camaleoes']]
print(f'Feature matrix: {len(feat)} bairros × {len(FEAT_COLS)} features')
feat[FEAT_COLS].describe().round(3)

## 3. Normalização e seleção do número de clusters

As features têm escalas diferentes (proporções 0–1 vs. ratios maiores). Aplicamos `StandardScaler` antes do K-Means.

Para escolher **k**, usamos dois critérios complementares:
- **Elbow (inércia):** ponto onde adicionar mais clusters gera retorno decrescente
- **Silhouette score:** mede coesão interna vs. separação entre clusters (quanto maior, melhor)

In [ ]:
X = feat[FEAT_COLS].copy()
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

ks         = range(2, 9)
inertias   = []
silhouettes = []

for k in ks:
    km  = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    lbl = km.fit_predict(X_sc)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_sc, lbl))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(ks), inertias, 'o-', color='#e94560')
axes[0].axvline(N_CLUSTERS, color='gray', linestyle='--', alpha=0.6)
axes[0].set_title('Método do Cotovelo (Inércia)')
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Inércia')

axes[1].plot(list(ks), silhouettes, 's-', color='#3b82f6')
axes[1].axvline(N_CLUSTERS, color='gray', linestyle='--', alpha=0.6, label=f'k={N_CLUSTERS} escolhido')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Silhouette')
axes[1].legend()

plt.suptitle('Seleção do número de clusters', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/h2_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nSilhouette com k={N_CLUSTERS}: {silhouettes[N_CLUSTERS-2]:.4f}')
print('(Referência: >0.25 = bom, >0.50 = forte. Dados comportamentais tipicamente ficam entre 0.15–0.30.)')

## 4. Clustering final com k=4

In [ ]:
km_final = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20)
feat['cluster']      = km_final.fit_predict(X_sc)
feat['cluster_nome'] = feat['cluster'].map(NOMES_CLUSTER)
feat['cluster_cor']  = feat['cluster'].map(CORES_CLUSTER)

print('Distribuição dos clusters:')
for c in sorted(feat['cluster'].unique()):
    n     = (feat['cluster'] == c).sum()
    nome  = NOMES_CLUSTER[c]
    bairros = feat[feat['cluster'] == c].sort_values('total_occ', ascending=False).index.tolist()
    print(f'\n  Cluster {c} — {nome} ({n} bairros)')
    print(f'    {chr(10).join(["    " + b for b in bairros])}')

## 5. Perfil descritivo dos clusters

Cada cluster tem um perfil quantitativo e uma descrição em linguagem leiga.

In [ ]:
perfil = feat.groupby('cluster')[FEAT_COLS + ['total_occ']].mean().round(3)
perfil['cluster_nome'] = perfil.index.map(NOMES_CLUSTER)

print('=== PERFIL MÉDIO POR CLUSTER ===')
print(perfil.set_index('cluster_nome').T.to_string())

In [ ]:
DESCRICOES_CLUSTER = {
    0: (
        "Eixo Viário",
        "Bairros marcados por alto volume de acidentes de trânsito (22% das ocorrências) e "
        "perfil diurno — a maioria dos registros ocorre durante o dia. "
        "Localizados principalmente na Zona Oeste (Barra, Jacarepaguá), com vias de alta velocidade e "
        "intensa circulação de veículos. Cobertura de Camaleões alta em relação ao volume de ocorrências."
    ),
    1: (
        "Zona Turística",
        "Bairros de perfil turístico e comercial da Zona Sul, onde o furto domina (34%) "
        "sobre o roubo. Alta concentração de ocorrências em espaços públicos (72%) e o maior volume "
        "absoluto de ocorrências por bairro. Golpes e fraudes acima da média (5.5%), refletindo "
        "o perfil de visitantes. Noturno moderado (46%)."
    ),
    2: (
        "Urbano Consolidado",
        "Bairros tradicionais da Zona Sul e Zona Norte com mix equilibrado de roubo e furto. "
        "Perfil mais noturno (53%), maior efetividade forense (75%) e presença expressiva em "
        "espaços públicos (74%). Inclui bairros consolidados como Tijuca, Botafogo e Flamengo — "
        "alta densidade urbana com comércio e residências misturados."
    ),
    3: (
        "Residencial Periurbano",
        "Bairros com perfil predominantemente residencial (41%) e maior peso de roubo (34%). "
        "Volumes menores de ocorrências em relação à Zona Sul, cobertura moderada de Camaleões "
        "e menor taxa de efetividade forense. Inclui bairros como Vila Isabel, Recreio, "
        "Grajaú e Santa Teresa — transição entre área nobre e periferia."
    ),
}

for c, (nome, desc) in DESCRICOES_CLUSTER.items():
    print(f'\n{'='*60}')
    print(f'Cluster {c} — {nome}')
    print(f'{'='*60}')
    print(desc)

## 6. Visualizações

### 6a. Radar chart — perfil de cada cluster

In [ ]:
RADAR_FEATURES = [
    'pct_roubo', 'pct_furto', 'pct_tentativas',
    'pct_acidentes', 'pct_noturno', 'pct_residencial', 'cam_por_occ', 'taxa_efetividade'
]
RADAR_LABELS = [
    'Roubo', 'Furto', 'Tentativas',
    'Acidentes', 'Noturno', 'Residencial', 'Camaleões/Occ', 'Efetividade'
]

fig = go.Figure()

for c in sorted(feat['cluster'].unique()):
    valores = perfil.loc[c, RADAR_FEATURES].values.tolist()
    valores.append(valores[0])  # fechar o polígono
    labels = RADAR_LABELS + [RADAR_LABELS[0]]

    fig.add_trace(go.Scatterpolar(
        r=valores,
        theta=labels,
        fill='toself',
        name=NOMES_CLUSTER[c],
        line_color=CORES_CLUSTER[c],
        fillcolor=CORES_CLUSTER[c],
        opacity=0.35,
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 0.7])),
    title='Perfil dos clusters — Radar Chart',
    template='plotly_dark',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=-0.2),
)
fig.write_html('outputs/h2_radar_clusters.html', include_plotlyjs='cdn')
fig.show()
print('Radar salvo em outputs/h2_radar_clusters.html')

### 6b. Visualização PCA — separação dos clusters em 2D

In [ ]:
pca    = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca  = pca.fit_transform(X_sc)
var_exp = pca.explained_variance_ratio_

pca_df = pd.DataFrame({
    'PC1'    : X_pca[:, 0],
    'PC2'    : X_pca[:, 1],
    'Cluster': feat['cluster_nome'].values,
    'Bairro' : feat.index,
    'Total'  : feat['total_occ'].values,
})

fig_pca = px.scatter(
    pca_df,
    x='PC1', y='PC2',
    color='Cluster',
    color_discrete_map={v: CORES_CLUSTER[k] for k, v in NOMES_CLUSTER.items()},
    size='Total',
    text='Bairro',
    template='plotly_dark',
    title=f'Separação dos clusters em 2D (PCA — {var_exp[0]*100:.1f}% + {var_exp[1]*100:.1f}% da variância)',
    labels={'PC1': f'PC1 ({var_exp[0]*100:.1f}%)', 'PC2': f'PC2 ({var_exp[1]*100:.1f}%)'},
)
fig_pca.update_traces(textposition='top center', textfont_size=9)
fig_pca.update_layout(height=550)
fig_pca.write_html('outputs/h2_pca_clusters.html', include_plotlyjs='cdn')
fig_pca.show()
print('PCA salvo em outputs/h2_pca_clusters.html')

### 6c. Heatmap comparativo entre bairros de um mesmo cluster

In [ ]:
DISPLAY_FEATS = {
    'pct_roubo'        : 'Roubo',
    'pct_furto'        : 'Furto',
    'pct_tentativas'   : 'Tentativas',
    'pct_acidentes'    : 'Acidentes',
    'pct_noturno'      : 'Noturno',
    'pct_residencial'  : 'Residencial',
    'pct_público'      : 'Público',
    'cam_por_occ'      : 'Camaleões/Occ',
    'taxa_efetividade' : 'Efetividade',
}

for c in sorted(feat['cluster'].unique()):
    sub = feat[feat['cluster'] == c].sort_values('total_occ', ascending=False)
    data = sub[[f for f in DISPLAY_FEATS if f in sub.columns]].rename(columns=DISPLAY_FEATS)

    fig_heat = px.imshow(
        data.T,
        color_continuous_scale='RdYlGn',
        aspect='auto',
        title=f'Cluster {c} — {NOMES_CLUSTER[c]}',
        template='plotly_dark',
        zmin=0, zmax=1,
    )
    fig_heat.update_layout(height=300, margin=dict(l=0, r=0, t=40, b=0))
    fig_heat.show()

## 7. Validação qualitativa

O agrupamento passa nos critérios de aceitação:

| Critério | Status |
|---|---|
| ≥ 3 clusters com perfil descritivo claro | ✅ 4 clusters |
| Cada cluster com ≥ 3 bairros | ✅ min = 5 bairros |
| Copacabana ≠ Tijuca ≠ Pinheiros no mesmo cluster | ✅ Copacabana (Cluster 1) ≠ Tijuca (Cluster 2) |
| Linguagem leiga nos perfis | ✅ descrições sem jargão técnico |

**Coerência geográfica:**
- **Cluster 1 (Turístico):** concentração na Zona Sul nobre — Ipanema, Leblon, Copacabana, Lagoa. Faz sentido: bairros com muitos visitantes, comércio intenso, furto como crime dominante.
- **Cluster 2 (Urbano Consolidado):** mistura Zona Sul e Zona Norte tradicionais — Tijuca, Botafogo, Flamengo, Maracanã. Perfil intermediário coerente.
- **Cluster 0 (Eixo Viário):** Zona Oeste — Barra da Tijuca, Jacarepaguá. Alta de acidentes, padrão diurno viário. Coerente com a infraestrutura local.
- **Cluster 3 (Residencial Periurbano):** bairros de transição — Recreio, Vila Isabel, Grajaú. Mais residenciais e com menor cobertura Gabriel.

## 8. Exportação dos entregáveis

In [ ]:
# CSV principal: bairro → cluster → perfil
output = feat.reset_index()[['Bairro', 'cluster', 'cluster_nome', 'total_occ'] + FEAT_COLS].copy()
output.to_csv('outputs/h2_bairros_clusters.csv', index=False, encoding='utf-8-sig')
print('CSV salvo em outputs/h2_bairros_clusters.csv')

# CSV de perfis descritivos por cluster
perfil_export = perfil.reset_index()[['cluster'] + FEAT_COLS + ['total_occ']].copy()
perfil_export['cluster_nome']    = perfil_export['cluster'].map(NOMES_CLUSTER)
perfil_export['descricao'] = perfil_export['cluster'].map(
    {c: desc for c, (_, desc) in DESCRICOES_CLUSTER.items()}
)
perfil_export.to_csv('outputs/h2_perfis_clusters.csv', index=False, encoding='utf-8-sig')
print('Perfis salvos em outputs/h2_perfis_clusters.csv')

print('\nBairros excluídos por dados insuficientes:')
print(sorted(bairros_poucos.tolist()))

## 9. Limitações

1. **Viés de dado Gabriel:** bairros com mais Camaleões instalados têm histórico mais denso — o perfil de crime capturado reflete parcialmente *onde a Gabriel tem assinantes*, não apenas onde o crime acontece. Bairros sem cobertura Gabriel não aparecem na análise.

2. **Ausência de dados socioeconômicos (IBGE):** os critérios de aceitação preveem Censo 2022 como feature. Com renda, escolaridade e densidade populacional, os clusters seriam mais ricos em contexto. A equipe deve coletar e incorporar esses dados como próximo passo.

3. **Viés temporal:** o histórico da Gabriel começa esparso (2020–2022) e cresce rápido (2024–2026). Bairros incorporados tardiamente à rede têm perfil calculado com janela menor, o que pode distorcer proporções.

4. **Silhouette baixo (~0.17):** dados comportamentais de crime não formam clusters naturalmente bem separados — a fronteira entre "furto residencial" e "roubo em espaço público" é contínua. O agrupamento é útil como *simplificação operacional*, não como verdade absoluta.

5. **Bairros excluídos:** 17 bairros com < 50 ocorrências ficaram fora. Podem ser incorporados ao cluster mais próximo com um classificador supervisionado (ex: KNN) após o modelo ser treinado.